# Случайный лес

In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
import joblib
import pickle
from sklearn.model_selection import train_test_split

In [3]:
df_load = pd.read_csv('data/drom_archive_cleaned_2018-2025full.csv')
df = df_load

```
Преобразование категориальных переменных
```

In [5]:
categorical = ['Тип двигателя', 'Коробка передач', 'Привод', 'Поколение', 'Тип кузова', 'Метка', 'Город']
numerical = ['Год', 'Объем двигателя', 'Мощность', 'Пробег', 'Год объявления', 'Возраст авто']

In [6]:
preprocessor = ColumnTransformer(transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), categorical), ('num', 'passthrough', numerical)], remainder='drop')

In [7]:
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        max_depth=20,
        verbose=2
    ))
])

```
Загрузка обучающей и тестовой выборок
```

In [8]:
y = df['Цена']
X = df.drop('Цена', axis=1)

In [9]:
df['price_stratum'] = pd.qcut(df['Цена'], q=10, labels=False)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=df['price_stratum'])

```
Обучение и сохранение модели
```

In [10]:
model.fit(X_train, y_train)
joblib.dump(model, "models/rf_model.pkl")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.


building tree 2 of 100
building tree 1 of 100
building tree 3 of 100
building tree 4 of 100



KeyboardInterrupt



```
Предсказание на тестовой выборке
```

In [ ]:
y_pred = model.predict(X_test)

```
Оценка модели
```

In [ ]:
rf_mse = mean_squared_error(y_test, y_pred)
rf_rmse = np.sqrt(rf_mse)
rf_mae = mean_absolute_error(y_test, y_pred)
rf_r2 = r2_score(y_test, y_pred)
rf_mape = mean_absolute_percentage_error(y_test, y_pred)

```
Вывод результатов
```

In [ ]:
pd.options.display.float_format = '{:_.2f}'.format
pd.DataFrame({
    'Метод оценки': ['Среднеквадратическая ошибка (MSE)', 'Среднеквадратическая ошибка (RMSE)', 'Средняя абсолютная ошибка (MAE)', 'Коэффицент детерминации (R^2)', 'Средняя абсолютная процентная ошибка (MAPE)'],
    'Результаты': [rf_mse, rf_rmse, rf_mae, rf_r2, rf_mape]
})

```
Сохранение метрик в отдельный файл
```

In [ ]:
metrics = {
    "model_name": "Random Forest",
    "MSE": rf_mse,
    "RMSE": rf_rmse,
    "MAE": rf_mae,
    "R2": rf_r2,
    "MAPE": rf_mape
}

In [ ]:
with open("metrics/rf_metrics.pkl", "wb") as f:
    pickle.dump(metrics, f)